# Where Factors Come From, and the Factor Zoo
## 🎯 Learning Objectives

By the end of today you will be able to:

1. **Distinguish the three kinds of factor model** — and say what you have to
   specify in each
2. **Name the major factor families** and the economic story behind each
3. **Say what a value ratio and a lead-lag signal are each betting on** — and
   why a spread in average returns is not enough to make a useful factor
4. **Measure how much two signals overlap** rather than guessing from their names
5. **Show that the factor zoo is far smaller than its headcount suggests**
6. **Explain why that matters** for anyone testing a new signal

## 📋 Today's Plan

1. [Three kinds of factor model](#three)
2. [The zoo: what people actually trade](#zoo)
3. [Where two factors come from](#where)
4. [Pitfall checklist](#pitfalls)
5. [🔄 Live Demo: how much do these overlap?](#demo)
6. [The zoo is smaller than it looks](#smaller)
7. [🛠️ Hands-On: find your signal's twin](#ho1)
8. [🎯 Challenge: does beta explain it?](#challenge) — *homework*
9. [Key takeaways](#takeaways)

---

## 🛠️ Setup

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [11, 4.5]
import warnings; warnings.filterwarnings('ignore')

BASE  = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"
panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")
menu  = pd.read_csv(f"{BASE}/signal_menu.csv")
ff    = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True)

print(f"{len(menu)} signals across {menu['Cat.Economic'].nunique()} economic categories")
menu.groupby('Cat.Economic').size().sort_values(ascending=False).to_string()

---

## 1. Three Kinds of Factor Model <a id="three"></a>

Last class we regressed Berkshire Hathaway on the market. Where did "the market" come from? We
chose it. That choice is the model.

There are exactly three ways to build a factor model, and they differ in **what
you have to specify**:

| Type | You **specify** | You **estimate** | How | Example |
|---|---|---|---|---|
| **Time-series** | the factor *returns* | the betas | time-series regression | Fama-French — HML is a portfolio return you can look up |
| **Characteristic** (fundamental) | the *betas* | the factor returns | cross-sectional regression, each period | BARRA — book-to-market **is** the loading |
| **Statistical** | **nothing** | both | eigendecomposition of Σ | PCA |

> **💡 Key Insight: the same duality you already met**
>
> In Lecture 3 you sorted stocks on a characteristic to *make* a portfolio. In
> Lecture 4 you regressed a return on a portfolio to *get* a beta. Those are the
> two directions:
>
> - Make a portfolio from a characteristic → you now have a **factor return** →
>   time-series model
> - Take the characteristic as the exposure directly → estimate what that
>   exposure paid each month → **characteristic model**
>
> Same raw material, opposite plumbing.

Today is about the first two, and specifically about **which** characteristics
are worth turning into factors. The statistical route gets its own lecture (L18)
because it answers a different question: what if the ones we chose are wrong?

---

## 2. The Zoo: What People Actually Trade <a id="zoo"></a>

Over fifty years the literature has published **300+** characteristics that
predict the cross-section of returns. They are not 300 different ideas. They
cluster into a handful of families, each with an economic story.

| Family | The claim | Canonical signal | Why it might work |
|---|---|---|---|
| **Value** | Cheap firms outperform | Book-to-market (Stattman 1980) | Risk of distress, or over-extrapolation of bad news |
| **Momentum** | Recent winners keep winning | 12-month return, skip a month (Jegadeesh-Titman 1993) | Under-reaction to news, then over-shoot |
| **Profitability** | Profitable firms outperform | Gross profits / assets (Novy-Marx 2013) | Quality is under-priced relative to growth |
| **Investment** | Firms that grow assets fast underperform | Asset growth (Cooper et al. 2008) | Empire-building; over-investment at the peak |
| **Low risk** | Low-volatility, low-beta stocks outperform | Idiosyncratic vol (Ang et al. 2006) | Leverage-constrained investors bid up high-beta names |
| **Issuance** | Firms issuing equity underperform | Share issuance (Pontiff-Woodgate 2008) | Managers issue when the stock is expensive |
| **Liquidity** | Illiquid stocks earn a premium | Amihud (2002) | Compensation for not being able to get out |
| **Lead-lag** | News about one firm shows up late in a linked firm | Customer momentum (Cohen-Frazzini 2008) | Limited attention: information travels slowly along economic links |

> **📌 Every one of these is in your signal menu**, with the original paper and
> the t-statistic its authors reported.



### Two stories, and they are not the same

Every family above has *two* competing explanations, and which one you believe
changes what you expect next.

**Risk.** The premium is compensation for bearing something genuinely
unpleasant. If so it should persist — nobody is going to arbitrage away payment
for real risk.

**Mispricing.** Investors are making a systematic error. If so, the premium
should **shrink once the paper is published** and people trade against it.

> **🤔 Hold this question.** You saw in Lecture 3 that the size premium went
> negative over 1980–2000, right after Banz published it in 1981. Which story
> does that support? We test this properly in Lecture 9.

Notice that only one side of that pair is doing any work here. "Mispricing"
makes a prediction we can check. "Risk" so far is just a label — nothing above
says *why* an investor would demand to be paid for holding cheap firms, or what
it is about bad times that makes them unwilling to.

That argument exists, and it is the thing that separates a factor from a
coincidence: **you should deviate from the market only if you are different from
the average investor.** If you fear recessions less than everyone else does, you
should hold more of whatever pays badly in recessions, and you should be paid
for it. If you are exactly average, you should hold the market and nothing else.

We build that argument out properly late in the term. Keep it in view until
then, because without it the list below is a list of things that happened to
work — and equilibrium reasoning is the main defence you have against
overfitting.

---

## 3. Where Two Factors Come From <a id="where"></a>

### Value: a price ratio has to predict something

$$P_0 = \frac{D_1}{r - g} \qquad\Longrightarrow\qquad \frac{D_1}{P_0} = r - g$$

A high dividend yield means investors expect a **high return** $r$ or **slow
cash-flow growth** $g$ — nothing else. That is what a price means if it is the
value of future cash flows. The other value ratios put a different number on top
(book, earnings, EBITDA):

$$\frac{X}{P_0} = \frac{X}{D_1}\,(r - g)$$

- A value sort is a bet on the $r$ part. If cheap firms are cheap because their
  cash flows really will shrink, it earns nothing.
- That is why value is run next to profitability (`GP`) and investment
  (`AssetGrowth`): they soak up the cash-flow part.
- The formula says $r$ is high, not *why*. Risk and mispricing both fit.

### Lead-lag: news travels slowly

Both signals use a price move in one firm to predict a later move in a **linked**
firm.

- **`CustomerMomentum`** (Cohen–Frazzini 2008). Firms disclose customers above
  10% of sales. Buy suppliers whose customers rose last month; short those whose
  customers fell.
- **`retConglomerate`** (Cohen–Lou 2012). Last month's return on single-industry
  firms in a conglomerate's industries, weighted by its sales mix.

The news is public and already in one price. It reaches the linked price late
because connecting the two takes attention, and attention is scarce. No identity
forces this and no obvious risk explains it, so it predicts: **short-lived**, and
**strongest where few people are watching**.

Top minus bottom decile, NYSE breakpoints, 1980–2000:

| | equal-weighted | value-weighted |
|---|---|---|
| `CustomerMomentum` | +15.9%/yr (t = 4.42) | +13.5%/yr (t = 2.63) |
| `retConglomerate` | +21.0%/yr (t = 5.49) | +6.8%/yr (t = 1.73) |

### A return spread is not yet a factor

$$R^{LS}_t = \alpha + \beta\, R^e_{m,t} + \varepsilon_t$$

Suppose high-signal stocks simply have higher betas: $\beta = 0.5$, market premium
8%. The long-short earns $0.5 \times 8\% = 4\%$ with $\alpha = 0$. It does sort
stocks by expected return — through market exposure. Half a unit of the market
earns the same 4% with less risk.

A portfolio moves the frontier only through its alpha (Performance Evaluation
lecture):

$$SR_{\max}^2 = SR_m^2 + \left(\frac{\alpha}{\sigma_\varepsilon}\right)^2$$

| Question | Test |
|---|---|
| Does the signal predict returns? | average top minus bottom $\neq 0$ |
| Does it add anything to the market? | $\alpha \neq 0$ |

It cuts both ways: value earned +5.2%/yr with $\beta = -0.20$, so its alpha was
+7.1%. Next lecture the benchmark gets more factors; the test stays alpha.

---

## 🛡️ Pitfall Checklist for Comparing Factors <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---|---|---|
| 1 | **Assuming different names mean different bets** | You "diversify" across five signals that are one signal | Correlate the long-short returns, not the characteristics |
| 2 | **Correlating the signals instead of the strategies** | Raw characteristics can look unrelated while their portfolios move together | Always correlate the *return series* |
| 3 | **Comparing over different samples** | Two signals with different data coverage aren't comparable | Align on common months before correlating |
| 4 | **Reading a category label as an economic claim** | "Liquidity" and "size" are different words for overlapping things | Check the number, not the taxonomy |
| 5 | **Counting signals as evidence** | 300 papers finding predictability is not 300 independent confirmations | How many *distinct* bets are there? — today |

> **🤖 AI-Era Insight**
>
> Ask an AI to "build a diversified multi-factor strategy" and it will happily
> combine value, size, low-volatility and liquidity — three of which, as you're
> about to see, are close to the same trade in this sample. It has the names,
> not the correlation matrix.

---

## 🔄 Live Demo: How Much Do These Overlap? <a id="demo"></a>

Thirty-two signals, each from a published paper. Are they thirty-two bets?

> **🤔 The question, and it is the whole lecture.** To find out whether two
> signals are the same thing, **what exactly do you correlate?** You have two
> candidates in front of you and they are not equivalent:
>
> - the **characteristics** — one number per stock per month, sitting in the
>   signal files
> - the **strategies** — one number per month, the long-short return each
>   signal produces
>
> Commit to one before you read on. And predict the average pairwise
> correlation while you're at it.


In [ ]:
# === YOUR TURN ===
MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below ----


In [ ]:
#@title 🔒 Reference implementation — `long_short()` and the full-menu build
panel['me_l1'] = panel.groupby('permno')['me'].shift(1)   # Lecture 2 convention

def long_short(sig):
    s = pd.read_parquet(f"{BASE}/signals/{sig}.parquet")
    d = panel.merge(s, on=['permno','date'], how='left').sort_values(['permno','date'])
    d['sig_l1'] = d.groupby('permno')[sig].shift(1)
    d = d.dropna(subset=['sig_l1', 'ret', 'me_l1'])
    q = (d[d.exchcd == 1].groupby('date')['sig_l1'].quantile([.1,.9]).unstack()
           .rename(columns={0.1:'lo', 0.9:'hi'}))
    d = d.merge(q, on='date')
    d['g'] = np.where(d.sig_l1 <= d.lo, 0, np.where(d.sig_l1 >= d.hi, 9, np.nan))
    d = d.dropna(subset=['g'])
    p = d.groupby(['date','g']).apply(lambda g: np.average(g['ret'], weights=g['me_l1'])).unstack()
    return (p[9] - p[0]).dropna()      # already dated by the month earned

SIGS = sorted(menu.Acronym)          # the menu IS the directory listing
R = {}
for s in SIGS:
    try:
        r = long_short(s)
        if len(r) > 200: R[s] = r
    except Exception:
        pass
L = pd.DataFrame(R).dropna(how='all')
print(f"{L.shape[1]} long-short strategies over {len(L)} common months")

In [ ]:
#@title 🔒 Check — run after the long_short cell below
# Three pairs, measured both ways.
def char_corr(a, b):
    x = pd.read_parquet(f"{BASE}/signals/{a}.parquet")
    y = pd.read_parquet(f"{BASE}/signals/{b}.parquet")
    j = x.merge(y, on=['permno','date'], how='inner').dropna()
    return j[a].corr(j[b])

print(f"{'pair':30s}{'CHARACTERISTICS':>18s}{'STRATEGIES':>13s}")
print("-"*61)
for a, b in [('MaxRet','RealizedVol'), ('Illiquidity','Size'), ('BM','EP')]:
    rc = pd.concat([long_short(a), long_short(b)], axis=1).dropna().corr().iloc[0,1]
    print(f"{a + ' & ' + b:30s}{char_corr(a,b):>18.2f}{rc:>13.2f}")

### Look at the middle row

**Illiquidity and Size correlate 0.07 as characteristics and 0.92 as
strategies.**

As raw numbers they look like unrelated ideas — Amihud's price-impact measure
and log market cap are barely related stock by stock. Sort on each, form the
long-shorts, and you are running **the same trade**. The liquidity premium and
the size premium are not two findings in this sample.

That is the answer to the question. **You correlate the strategies.** The
characteristics tell you how the numbers are distributed; only the portfolios
tell you what you are holding — and a portfolio is what you own.

> **🤖 AI-Era Insight**
>
> "Correlate these signals" is the natural request and the natural code is
> `pivot` then `.corr()` on the characteristics. It runs in one line, and on the
> middle row it would have told you those two strategies were independent.


In [ ]:
C  = L.corr()
iu = np.triu_indices_from(C, 1)
v  = C.values[iu]

print(f"pairwise correlations across {L.shape[1]} strategies\n")
print(f"  mean      {v.mean():+.3f}")
print(f"  median    {np.median(v):+.3f}")
print(f"  |corr|>0.5 in {(np.abs(v)>0.5).mean():.1%} of pairs\n")

names = C.columns
pairs = sorted((C.values[i,j], names[i], names[j]) for i,j in zip(*iu))
print("Most POSITIVELY correlated pairs:")
for c,a,b in pairs[-5:][::-1]: print(f"  {a:22s} {b:22s} {c:+.2f}")
print("\nMost NEGATIVELY correlated pairs:")
for c,a,b in pairs[:3]:        print(f"  {a:22s} {b:22s} {c:+.2f}")

In [ ]:
order = (menu.set_index('Acronym').loc[[c for c in C.columns], 'Cat.Economic']
           .sort_values().index.tolist())
fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(C.loc[order, order], cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=7)
ax.set_title(f'Correlation of {len(order)} long-short strategies, sorted by economic category',
             fontweight='bold')
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

### Read the average first, then the picture

The **average pairwise correlation is about +0.04** — essentially zero. Taken at
face value that says the zoo is wonderfully diversified: thirty-one nearly
independent sources of return.

That is the wrong conclusion, and the heatmap shows why. Look at the blocks on
the diagonal.

---

## 4. The Zoo Is Smaller Than It Looks <a id="smaller"></a>

The average hides the structure. Split the pairs by whether the two signals come
from the **same economic category**.

In [ ]:
cat = menu.set_index('Acronym')['Cat.Economic'].to_dict()
within, across = [], []
for i, j in zip(*iu):
    a, b = names[i], names[j]
    (within if cat.get(a) == cat.get(b) else across).append(abs(C.values[i,j]))

print(f"mean |correlation|\n")
print(f"  WITHIN economic category   {np.mean(within):.3f}   (n = {len(within)} pairs)")
print(f"  ACROSS economic category   {np.mean(across):.3f}   (n = {len(across)} pairs)")
print(f"\n  ratio: {np.mean(within)/np.mean(across):.1f}x")

### Three papers, one factor

Look at the top of the correlation list:

| pair | correlation |
|---|---|
| MaxRet & RealizedVol | **+0.97** |
| IdioVol3F & RealizedVol | **+0.96** |
| IdioVol3F & MaxRet | **+0.94** |
| Illiquidity & Size | **+0.92** |

Those are separate publications, in separate journals, with separate names and
separate economic stories. At a correlation of 0.97 they are **the same trade**.
Whatever is being paid for, all three are collecting it.

And Illiquidity vs Size at 0.92: illiquid firms *are* small firms. The
"liquidity premium" and the "size premium" are not two findings in this sample.

> **💡 Key Insight: count bets, not papers**
>
> The zoo has 300+ entries and far fewer distinct positions. Signals inside a
> family are near-duplicates (mean |ρ| ≈ 0.56); signals across families are
> nearly independent (≈ 0.18).
>
> **Diversification comes from spanning families, not from collecting names.**

> **⚠️ Caution: this breaks a lot of published evidence**
>
> If 300 papers each report an independent-looking discovery, that seems like
> overwhelming support for cross-sectional predictability. If those 300 papers
> are really testing eight or ten distinct bets over and over on overlapping
> samples, it is much weaker support than the headcount suggests.
>
> That is one of the two big problems with the anomaly literature. We take it
> apart in Lecture 9, and Lecture 18 asks the sharper version: how many distinct
> factors can this data support *at all*?

---

## 🛠️ Hands-On: Find Your Signal's Twin <a id="ho1"></a>

You picked a signal in Lecture 3 and regressed it in Lecture 4. Now find out
what else it is.

### Your task

Pull your signal's row out of the correlation matrix and look at its nearest
neighbours. Then check whether they're in the same economic category — if the
nearest neighbour is in a *different* family, that's worth knowing.

In [ ]:
# === EDIT + YOUR TURN ===
MY_SIGNAL = "GP"        # ← your pick from Lecture 3

row = ____              # hint: C[MY_SIGNAL].drop(MY_SIGNAL).sort_values(ascending=False)

print(f"{MY_SIGNAL}  ({cat.get(MY_SIGNAL)})\n")
print("closest 5:")
for s, c in row.head(5).items():
    flag = "  ← same family" if cat.get(s) == cat.get(MY_SIGNAL) else ""
    print(f"  {s:24s} {c:+.2f}   {cat.get(s):22s}{flag}")
print("\nmost negatively correlated:")
for s, c in row.tail(3).items():
    print(f"  {s:24s} {c:+.2f}   {cat.get(s)}")

### What to take from it

If your signal's nearest neighbour is above **+0.8**, you are not holding a
distinct bet — you're holding a version of something else, and any "confirmation"
from that other signal is not independent evidence.

If your closest neighbour is below **+0.3**, your signal is doing something the
rest of the zoo isn't. That is more interesting, and also more suspicious: check
that it isn't just noisier.

---

## 🎯 Challenge: Does Beta Explain It? <a id="challenge"></a>

*Homework — due before Lecture 6.*

Two signals: `BookLeverage` and `IdioVol3F`. For each, find out whether its spread
in average returns is anything more than a spread in beta — and what that means
for someone who already holds the market.

### Q1 — Ten portfolios, their betas, and the long-short

For each signal, sort the way this lecture's `long_short()` does — signal lagged
one month, NYSE breakpoints, value-weighted by `me_l1` — but put breakpoints at
every tenth percentile (10th, 20th, …, 90th) and keep **all ten** portfolios.
Every stock goes into the portfolio its signal falls in, NYSE or not.

Subtract the T-bill rate (`ff['RF']`) from each portfolio's return and regress it
on the market's excess return (`ff['Mkt-RF']`), 1980–2000.

Then form the long-short, portfolio 10 minus portfolio 1, and regress it on the
market too. The T-bill rate cancels in a long-short, so do not subtract it again.

Write it once as a function and run it on both signals.

> **📌 Required variable names**, for each signal (`_bl` for BookLeverage, `_iv`
> for IdioVol3F):
> ```python
> betas_bl   = ____   # 10 values: market beta of portfolios 1 to 10
> avg_bl     = ____   # 10 values: average excess return, annualized
> ls_beta_bl = ____   # 10 - 1: market beta
> ls_avg_bl  = ____   # 10 - 1: average return, annualized
> ls_t_bl    = ____   # 10 - 1: t-statistic of its alpha
> ```

In [ ]:
# Your work here


# Required outputs — fill these in:
betas_bl   = ____
avg_bl     = ____
ls_beta_bl = ____
ls_avg_bl  = ____
ls_t_bl    = ____

betas_iv   = ____
avg_iv     = ____
ls_beta_iv = ____
ls_avg_iv  = ____
ls_t_iv    = ____

for name, b, a, lb, la, lt in [('BookLeverage', betas_bl, avg_bl, ls_beta_bl, ls_avg_bl, ls_t_bl),
                               ('IdioVol3F',    betas_iv, avg_iv, ls_beta_iv, ls_avg_iv, ls_t_iv)]:
    print(name)
    print(pd.DataFrame({'beta': np.asarray(b), 'avg excess return': np.asarray(a)},
                       index=range(1, 11)).round(3).to_string())
    print(f"  10 - 1:  beta {lb:+.2f}   average {la:+.2%}/yr   alpha t = {lt:+.2f}\n")

### Q2 — The picture

Two panels, one per signal. In each, plot beta on the x-axis against average
excess return on the y-axis for the ten portfolios, labelled 1 to 10. Add the
**long-short** 10 − 1 as a different marker, and the **market**. What is the
market's beta?

> **📌 Required variable name:**
> ```python
> mkt_avg = ____   # the market's average excess return, annualized, same months
> ```

In [ ]:
# Required output — fill this in:
mkt_avg = ____

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Your plot here: in each panel the ten portfolios (labelled 1-10),
# the 10 - 1 long-short, and the market


for ax, name in zip(axes, ['BookLeverage', 'IdioVol3F']):
    ax.set_title(name); ax.set_xlabel('market beta')
axes[0].set_ylabel('average excess return, annualized')
plt.tight_layout(); plt.show()

### Q3 — The memo

> **📝 Your task — maximum 6 sentences**
>
> Describe what you see in each plot. Do average returns line up with beta?
> Where does each long-short sit?
>
> Then: an investor is fully invested in the market and wants a higher Sharpe
> ratio. What does each plot tell them to do?
>
> *Hint: what could that investor already get by holding more or less of the
> market — or even shorting it — and keeping the rest in T-bills? Where would
> that put them on your plots?*

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["betas_bl", "avg_bl", "ls_beta_bl", "ls_avg_bl", "ls_t_bl",
            "betas_iv", "avg_iv", "ls_beta_iv", "ls_avg_iv", "ls_t_iv",
            "mkt_avg", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")
lists = ["betas_bl", "avg_bl", "betas_iv", "avg_iv"]
if any(len(np.ravel(eval(k))) != 10 for k in lists):
    raise ValueError("\n❌ betas_* and avg_* should each hold ten numbers, portfolios 1 to 10")

payload = {
    "assignment": "L5_FactorZoo_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: ([float(x) for x in np.ravel(eval(k))] if k in lists else float(eval(k)))
                for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")

---

## 🧠 Key Takeaways <a id="takeaways"></a>

1. **Three kinds of factor model**, distinguished by what you specify:
   time-series (you name the factor returns), characteristic (you name the
   exposures), statistical (you name nothing).

2. **A sort makes a factor; a regression consumes one.** Same raw material,
   opposite direction.

3. **The zoo has families, not 300 ideas** — value, momentum, profitability,
   investment, low-risk, issuance, liquidity, lead-lag.

4. **Every family has two stories, risk and mispricing**, and they predict
   different futures. Mispricing implies the premium dies after publication.

5. **A spread in average returns is not a new factor.** It adds something only
   if its alpha against the factors you already hold is nonzero:
   $SR_{\max}^2 = SR_m^2 + (\alpha/\sigma_\varepsilon)^2$.

6. **Correlate the strategies, never the signals.** Names and categories are not
   evidence.

7. **Within a family, mean |ρ| ≈ 0.56. Across families, ≈ 0.18.** Diversification
   comes from spanning families.

8. **Some "different" factors are the same trade.** MaxRet, RealizedVol and
   IdioVol3F correlate 0.94–0.97. Illiquidity and Size, 0.92.

9. **300 papers is not 300 pieces of evidence.** Which is a serious problem for
   the literature, and the subject of Lecture 9.

---

### Next class

We've been using one factor — the market. Next: how to run and read a model with
several, why alpha shrinks every time you add one, and the two different ways to
estimate the whole thing.

---

## 📎 Appendix <a id="appendix"></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📎 APPENDIX — data
# ═══════════════════════════════════════════════════════════════════════
# Everything today comes from the repo:
#   panel  = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")
#   signal = pd.read_parquet(f"{BASE}/signals/<Acronym>.parquet")
#   menu   = pd.read_csv(f"{BASE}/signal_menu.csv")
#
# The 32 signals are replications from Open Source Asset Pricing
# (Chen & Zimmermann), which covers 300+ published cross-sectional predictors:
#     https://www.openassetpricing.com
# `signal_menu.csv` carries each one's authors, year, journal, economic
# category, and the t-statistic the ORIGINAL paper reported — which is what
# makes the replication comparison in Lecture 3 possible.
